# Fine-tune MuRIL on the Kavach splits (Google Colab)

Fine-tunes [`google/muril-base-cased`](https://huggingface.co/google/muril-base-cased) (a BERT
pretrained on 17 Indian languages, including transliterated Hindi) on **the same splits** as the
TF-IDF baseline (`ml/train_classifier.py`), on **raw text** (no masking), and reports **the same
metrics** as `ml/eval_classifier.py`, so the two can be compared row for row.

Not integrated into the backend: a 240M-parameter model does not fit Render's 512 MB instance.
If it clearly beats the baseline, the next step is ONNX export + int8 quantization.

**Runtime → Change runtime type → T4 GPU.** About 15 minutes for 3 epochs.

## Data

Run `uv run python -m ml.prepare_dataset` locally, then upload from `backend/data/datasets/processed/`:
`train.csv`, `val.csv`, `test.csv`, `heldout.csv`, `ood_uci_unseen.csv`.

⚠️ `train.csv`, `val.csv`, `test.csv` and `heldout.csv` contain your collected messages (anonymized,
but still yours). Set `DROP_COLLECTED = True` below to leave them out before training, and delete
the files from the Colab session when done.

In [ ]:
!pip -q install "transformers>=4.44" "accelerate>=0.33" scikit-learn pandas

In [ ]:
import os

from google.colab import files

DROP_COLLECTED = False  # True: never train or evaluate on your own collected messages
NEEDED = ["train.csv", "val.csv", "test.csv", "heldout.csv", "ood_uci_unseen.csv"]
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print("upload:", missing)
    files.upload()

In [ ]:
import numpy as np
import pandas as pd

CLASSES = ["genuine", "promo_spam", "scam"]  # same order as the baseline
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}


def load(name):
    df = pd.read_csv(name, dtype=str, keep_default_na=False)
    if DROP_COLLECTED:
        df = df[df["dataset"] != "collected"]
    return df


train, val, test, heldout = (load(f) for f in NEEDED[:4])
uci = load("ood_uci_unseen.csv")
uci["label"] = np.where(uci["label"] == "spam", "spam", "genuine")
train = train[train["label"].isin(CLASSES)]
val = val[val["label"].isin(CLASSES)]
is_indian = heldout["is_indian"] == "true"
SETS = {
    "indian_test": test,
    "india_heldout": heldout[is_indian],
    "mendeley_heldout": heldout[heldout["dataset"] == "mendeley_sms_phishing"],
    "imc_heldout": heldout[(heldout["dataset"] == "imc25_smishing") & ~is_indian],
    "uci_unseen": uci,
}
print(
    "train",
    train["label"].value_counts().to_dict(),
    "synthetic",
    (train["is_synthetic"] == "true").sum(),
)
print("val", val["label"].value_counts().to_dict())
for k, v in SETS.items():
    print(k, len(v), v["label"].value_counts().to_dict())

## Sample weights: the same as the baseline

Class balance × template cap (a template group with more than 10 train messages counts as 10)
× 5 for Indian rows. See `sample_weights` in `ml/train_classifier.py`.

In [ ]:
GROUP_CAP, INDIAN_WEIGHT = 10, 5.0


def sample_weights(df):
    group = df["group"].where(df["group"] != "", df["id"])
    sizes = group.map(group.value_counts())
    w = np.minimum(1.0, GROUP_CAP / sizes.to_numpy(float))
    w = w * np.where(df["is_indian"] == "true", INDIAN_WEIGHT, 1.0)
    labels = df["label"].to_numpy()
    for c in CLASSES:
        m = labels == c
        if m.any():
            w[m] *= len(df) / (len(CLASSES) * w[m].sum())
    return w


train_w = sample_weights(train)

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

MODEL = "google/muril-base-cased"
MAX_LEN = 128
tok = AutoTokenizer.from_pretrained(MODEL)


class Messages(Dataset):
    def __init__(self, df, weights=None):
        self.enc = tok(df["text"].tolist(), truncation=True, max_length=MAX_LEN)
        self.labels = [LABEL2ID.get(lab, 0) for lab in df["label"]]
        self.weights = weights

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        if self.weights is not None:
            item["weight"] = float(self.weights[i])
        return item


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        weight = inputs.pop("weight", None)
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, reduction="none")
        if weight is not None:
            loss = (loss * weight.to(loss.dtype)).sum() / weight.sum()
        else:
            loss = loss.mean()
        return (loss, out) if return_outputs else loss


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(CLASSES), id2label=dict(enumerate(CLASSES)), label2id=LABEL2ID
)

In [ ]:
from sklearn.metrics import f1_score


def compute_metrics(p):
    return {"macro_f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")}


class Collator(DataCollatorWithPadding):
    def __call__(self, features):
        weights = [f.pop("weight") for f in features if "weight" in f]
        batch = super().__call__(features)
        if weights:
            batch["weight"] = torch.tensor(weights)
        return batch


args = TrainingArguments(
    output_dir="muril-kavach",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=0,
    remove_unused_columns=False,
)
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=Messages(train, train_w),
    eval_dataset=Messages(val),
    data_collator=Collator(tok),
    compute_metrics=compute_metrics,
)
trainer.train()

## Calibrate on val (temperature scaling, like the baseline)

In [ ]:
def logits_of(df):
    return trainer.predict(Messages(df)).predictions.astype(np.float64)


def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


val_logits = logits_of(val)
y_val = val["label"].map(LABEL2ID).to_numpy()


def nll(t):
    p = softmax(val_logits / t)
    return -np.log(p[np.arange(len(y_val)), y_val] + 1e-12).mean()


grid = np.exp(np.linspace(np.log(0.2), np.log(5.0), 161))
T = float(min(grid, key=nll))
print("temperature", T, "val macro-F1", f1_score(y_val, val_logits.argmax(1), average="macro"))

## The same metrics as `ml/eval_classifier.py`

Per-class precision/recall/F1 and confusion matrix (3 classes), scam-vs-rest precision, recall,
FPR (flagged = P(scam) ≥ 0.5), average precision, FPR at 90% scam recall. 95% Wilson intervals.

In [ ]:
import math


def wilson(k, n, z=1.96):
    if n == 0:
        return (math.nan, math.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0, c - h), min(1, c + h)


def rate(k, n):
    if n == 0:
        return "– (n=0)"
    lo, hi = wilson(k, n)
    return f"{k / n:.1%} ({k}/{n}, CI {lo:.0%}–{hi:.0%})"


def average_precision(s, y):
    if y.sum() == 0:
        return math.nan
    o = np.argsort(-s, kind="stable")
    s, t = s[o], y[o]
    tp = np.cumsum(t)
    last = np.r_[np.diff(s) != 0, True]
    prec = tp[last] / np.arange(1, len(s) + 1)[last]
    rec = tp[last] / y.sum()
    return float(np.sum(np.diff(np.r_[0.0, rec]) * prec))


def fpr_at_recall(s, y, target=0.9):
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return None
    for th in np.unique(s)[::-1]:
        if (pos >= th).mean() >= target:
            return float((neg >= th).mean()), float(th)


results = {}
for name, df in SETS.items():
    probs = softmax(logits_of(df) / T)
    pred = np.array(CLASSES)[probs.argmax(1)]
    labels = df["label"].to_numpy()
    y = np.isin(labels, ["scam", "spam"]).astype(int)
    s = probs[:, LABEL2ID["scam"]]
    flagged = s >= 0.5
    tp, fp = int((flagged & (y == 1)).sum()), int((flagged & (y == 0)).sum())
    fn, tn = int((~flagged & (y == 1)).sum()), int((~flagged & (y == 0)).sum())
    at = fpr_at_recall(s, y)
    print(f"\n=== {name} ({len(df)})")
    print(
        f"scam vs rest: precision {tp / max(1, tp + fp):.1%} · recall {rate(tp, tp + fn)} · "
        f"FPR {rate(fp, fp + tn)} · AP {average_precision(s, y):.1%} · FPR@90%R "
        f"{'–' if at is None else f'{at[0]:.1%}'}"
    )
    if name != "uci_unseen":
        print(pd.crosstab(pd.Series(labels, name="true"), pd.Series(pred, name="pred")))
        for c in CLASSES:
            sup, npred = (labels == c).sum(), (pred == c).sum()
            hit = ((labels == c) & (pred == c)).sum()
            if sup:
                p_, r_ = hit / max(1, npred), hit / sup
                f1_ = 2 * p_ * r_ / max(1e-9, p_ + r_)
                print(f"  {c:<11} support {sup:>4}  P {p_:.1%}  R {r_:.1%}  F1 {f1_:.1%}")
    else:
        print("UCI 'spam' is not our scam: read the FPR on ham only.")
    results[name] = {
        "n": len(df),
        "ap": average_precision(s, y),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "fpr_at_90_recall": None if at is None else at[0],
    }

In [ ]:
import json

json.dump(
    {"model": MODEL, "temperature": T, "results": results},
    open("muril_results.json", "w"),
    indent=2,
)
files.download("muril_results.json")
# Keep the fine-tuned weights if the numbers are worth it (≈950 MB):
# trainer.save_model("muril-kavach-best")